In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer    
from sklearn.metrics import mean_squared_error,f1_score,recall_score,accuracy_score,precision_score
from sklearn.base import BaseEstimator, TransformerMixin

In [2]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [3]:
test_df_1 = test_df.copy()

In [4]:
train_df.head()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,composition_label_0,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,...,time_signature_0,duration_ms_1,harmonic_scale_0,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0,target
0,76339,0.482850,1.169231,80.018,0.0201,Country Stuff (feat. Jake Owen),1.0,0.789,154586.0,NaN,...,4.0,161853.0,7.0,4.0,NaN,0.607,7.0,0.7250,0.000000,74
1,80006,0.267862,1.321321,147.966,0.3340,Solitude,6.0,0.715,46874.0,15.0,...,4.0,155619.0,1.0,4.0,0.843,0.783,4.0,NaN,0.043200,2
2,83501,0.242606,1.285319,142.980,0.1110,BDFFRNT (Saved from Conformity),4.0,NaN,264665.0,7.0,...,4.0,209378.0,6.0,4.0,NaN,0.211,10.0,0.6020,0.000000,35
3,81530,0.426400,1.279435,123.063,0.1960,Headlights (feat. Ilsey),5.0,0.685,209208.0,5.0,...,4.0,219043.0,11.0,4.0,0.702,0.369,NaN,0.8200,0.000335,70
4,60534,0.000000,0.974906,132.722,0.0811,Afraid,6.0,0.856,215346.0,5.0,...,4.0,258893.0,1.0,0.0,0.000,0.631,1.0,0.0221,0.000000,78


In [5]:
train_df.shape

(61609, 62)

In [6]:
train_df.columns

Index(['id', 'emotional_charge_2', 'groove_efficiency_1', 'beat_frequency_1',
       'organic_texture_2', 'composition_label_0', 'harmonic_scale_1',
       'intensity_index_0', 'duration_ms_0', 'album_name_length',
       'beat_frequency_0', 'beat_frequency_2', 'artist_count',
       'composition_label_1', 'publication_timestamp', 'weekday_of_release',
       'album_component_count', 'emotional_charge_1', 'emotional_charge_0',
       'tonal_mode_2', 'key_variety', 'performance_authenticity_2',
       'performance_authenticity_0', 'season_of_release', 'time_signature_1',
       'duration_ms_2', 'lunar_phase', 'instrumental_density_2',
       'organic_texture_0', 'creator_collective', 'vocal_presence_2',
       'tonal_mode_1', 'vocal_presence_1', 'vocal_presence_0',
       'intensity_index_1', 'organic_immersion_0', 'tonal_mode_0',
       'groove_efficiency_2', 'instrumental_density_1', 'organic_immersion_2',
       'duration_consistency', 'composition_label_2', 'organic_texture_1',
    

In [7]:
train_df.describe()

,id,emotional_charge_2,groove_efficiency_1,beat_frequency_1,organic_texture_2,harmonic_scale_1,intensity_index_0,duration_ms_0,album_name_length,beat_frequency_0,...,time_signature_0,duration_ms_1,harmonic_scale_0,time_signature_2,rhythmic_cohesion_2,emotional_resonance_0,harmonic_scale_2,intensity_index_2,instrumental_density_0,target
count,61609.000000,59167.000000,61429.000000,61223.000000,61226.000000,58304.000000,55638.000000,6.032000e+04,52015.000000,51878.000000,...,59704.000000,5.250400e+04,53925.000000,58455.000000,56049.000000,60063.000000,57142.000000,60916.000000,60900.000000,61609.000000
mean,51390.780162,0.316976,1.238856,121.022910,0.274748,5.192594,0.604426,2.011315e+05,18.225723,119.133973,...,3.874849,2.110477e+05,5.212499,3.901274,0.612252,0.458851,5.288894,0.616045,0.148391,52.067328
std,29659.344472,0.212777,6.171617,30.467061,0.303020,3.629153,0.243943,1.100738e+05,14.404713,32.067971,...,0.564558,8.911099e+04,3.571288,0.465295,0.179591,0.261196,3.567118,0.230109,0.306915,21.569248
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.485000e+03,1.000000,0.000000,...,0.000000,4.120000e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,25832.000000,0.143877,0.730914,96.138000,0.027125,2.000000,0.447000,1.482340e+05,9.000000,94.802250,...,4.000000,1.682000e+05,2.000000,4.000000,0.506000,0.243000,2.000000,0.477000,0.000000,37.000000
50%,51410.000000,0.291060,1.004894,120.012000,0.141000,5.000000,0.633000,1.959215e+05,14.000000,119.893000,...,4.000000,2.029730e+05,5.000000,4.000000,0.630000,0.449000,5.000000,0.646000,0.000019,57.000000
75%,77069.000000,0.466860,1.358251,141.401000,0.454000,8.000000,0.803000,2.402488e+05,23.000000,140.023000,...,4.000000,2.413605e+05,8.000000,4.000000,0.745000,0.663000,8.000000,0.791000,0.024600,69.000000
max,102681.000000,0.976063,654.000000,239.983000,0.996000,11.000000,1.000000,3.664274e+06,199.000000,235.998000,...,5.000000,3.550973e+06,11.000000,5.000000,0.979000,1.000000,11.000000,1.000000,1.000000,100.000000


In [8]:
print(train_df.isnull().sum().sort_values(ascending=False))

tempo_volatility       10417
beat_frequency_0        9731
album_name_length       9594
duration_ms_1           9105
creator_collective      8914
                       ...  
groove_efficiency_1      180
organic_immersion_1      138
vocal_presence_0         110
id                         0
target                     0
Length: 62, dtype: int64


In [9]:
test_df.isnull().sum()

id                           0
emotional_charge_2        1597
groove_efficiency_1        101
beat_frequency_1           279
organic_texture_2          228
                          ... 
rhythmic_cohesion_2       3682
emotional_resonance_0     1005
harmonic_scale_2          2931
intensity_index_2          432
instrumental_density_0     500
Length: 61, dtype: int64

In [10]:
test_df.columns

Index(['id', 'emotional_charge_2', 'groove_efficiency_1', 'beat_frequency_1',
       'organic_texture_2', 'composition_label_0', 'harmonic_scale_1',
       'intensity_index_0', 'duration_ms_0', 'album_name_length',
       'beat_frequency_0', 'beat_frequency_2', 'artist_count',
       'composition_label_1', 'publication_timestamp', 'weekday_of_release',
       'album_component_count', 'emotional_charge_1', 'emotional_charge_0',
       'tonal_mode_2', 'key_variety', 'performance_authenticity_2',
       'performance_authenticity_0', 'season_of_release', 'time_signature_1',
       'duration_ms_2', 'lunar_phase', 'instrumental_density_2',
       'organic_texture_0', 'creator_collective', 'vocal_presence_2',
       'tonal_mode_1', 'vocal_presence_1', 'vocal_presence_0',
       'intensity_index_1', 'organic_immersion_0', 'tonal_mode_0',
       'groove_efficiency_2', 'instrumental_density_1', 'organic_immersion_2',
       'duration_consistency', 'composition_label_2', 'organic_texture_1',
    

Feature Engineering

In [11]:
def feature_engineering(df):
    df = df.copy()
    
    drop_cols = ['id', 'track_identifier']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    
    if 'publication_timestamp' in df.columns:
        df['publication_timestamp'] = pd.to_datetime(df['publication_timestamp'], errors='coerce')
        
        df['year'] = df['publication_timestamp'].dt.year
        df['month'] = df['publication_timestamp'].dt.month
        
        df = df.drop(columns=['publication_timestamp'])
    
    groups = [
        'emotional_charge',
        'groove_efficiency',
        'beat_frequency',
        'organic_texture',
        'harmonic_scale',
        'intensity_index',
        'duration_ms',
        'tonal_mode',
        'performance_authenticity',
        'instrumental_density',
        'vocal_presence',
        'organic_immersion',
        'rhythmic_cohesion',
        'emotional_resonance',
        'time_signature'
    ]
    
    for g in groups:
        cols = [f"{g}_0", f"{g}_1", f"{g}_2"]
        existing = [c for c in cols if c in df.columns]
        
        if len(existing) > 0:
            df[f"{g}_mean"] = df[existing].mean(axis=1)    
            df = df.drop(columns=existing)
    
    return df

In [12]:
train_df = feature_engineering(train_df)
test_df = feature_engineering(test_df)

In [13]:
test_df.shape

(41074, 30)

In [14]:
train_df.shape

(61609, 31)

In [15]:
train_df.columns

Index(['composition_label_0', 'album_name_length', 'artist_count',
       'composition_label_1', 'weekday_of_release', 'album_component_count',
       'key_variety', 'season_of_release', 'lunar_phase', 'creator_collective',
       'duration_consistency', 'composition_label_2', 'tempo_volatility',
       'target', 'year', 'month', 'emotional_charge_mean',
       'groove_efficiency_mean', 'beat_frequency_mean', 'organic_texture_mean',
       'harmonic_scale_mean', 'intensity_index_mean', 'duration_ms_mean',
       'tonal_mode_mean', 'performance_authenticity_mean',
       'instrumental_density_mean', 'vocal_presence_mean',
       'organic_immersion_mean', 'rhythmic_cohesion_mean',
       'emotional_resonance_mean', 'time_signature_mean'],
      dtype='object')

Seperate feature and Target

In [16]:
X = train_df.drop('target', axis=1)
y = train_df['target']

 Identify Column Types

In [17]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

Preprocessing 

In [18]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

Train model using pipeline

In [19]:
reg_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

reg_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['album_name_length', 'artist_count', 'album_component_count',
       'key_variety', 'duration_consistency', 'tempo_volatility', 'year',
       'month', 'emotional_charge_mean', 'groove_efficiency_mean',
       'beat_frequency_me...
       'time_signature_mean'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['composition_label_0', 'composition_label_1', 'weekday_of_release',
       'season_of_release', 'lunar_phase', 'creator_collective',
       'composition_label_2'],
      dtype='object'))])),
                ('model', LinearRegression())])

In [45]:
y_pred = reg_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
accuracy = accuracy_score(y_test, y_pred.round())
f1score = f1_score(y_test, y_pred.round(), average='weighted')
precison = precision_score(y_test, y_pred.round(), average='weighted')
recall = recall_score(y_test, y_pred.round(), average='weighted')

print(f'RMSE: {rmse}')
print(f'Accuracy: {accuracy}')
print(f'F1 Score: {f1score}')
print(f'Precision: {precison}')
print(f'Recall: {recall}')

RMSE: 10.729172294388842
Accuracy: 0.4776821944489531
F1 Score: 0.48408752421698886
Precision: 0.5361370760194977
Recall: 0.4776821944489531


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [21]:
print("Min prediction:", y_pred.min())
print("Max prediction:", y_pred.max())

Min prediction: -3.810983152739979
Max prediction: 100.52627678311694


In [22]:
y_pred = np.clip(y_pred, 0, 100)

In [23]:
print("Min prediction:", y_pred.min())
print("Max prediction:", y_pred.max())

Min prediction: 0.0
Max prediction: 100.0


In [24]:
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(10.728998905503033)

Get predictions for test data

In [25]:
test_pred = reg_model.predict(test_df)

In [26]:
test_pred = np.clip(test_pred, 0, 100)

In [27]:
print("Min prediction:", test_pred.min())
print("Max prediction:", test_pred.max())

Min prediction: 0.0
Max prediction: 100.0


Submission

In [28]:
submission_df = pd.DataFrame({
    'id' : test_df_1['id'],
    'target' : test_pred
})
submission_df.to_csv('submission.csv', index=False)

Lasso Regression Model

In [47]:
lasso_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Lasso(alpha=0.1, random_state=42))
])

lasso_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['album_name_length', 'artist_count', 'album_component_count',
       'key_variety', 'duration_consistency', 'tempo_volatility', 'year',
       'month', 'emotional_charge_mean', 'groove_efficiency_mean',
       'beat_frequency_me...
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['composition_label_0', 'composition_label_1', 'weekday_of_release',
       'season_of_release', 'lunar_phase', 'creator_collective',
       'composition_label_2'],
      dtype='object'))])),
                ('model', Lasso(alpha=0.1, random_state=42))])

In [48]:
y_pred_lasso = lasso_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
accuracy = accuracy_score(y_test, y_pred_lasso.round())
f1score = f1_score(y_test, y_pred_lasso.round(), average='weighted')
precison = precision_score(y_test, y_pred_lasso.round(), average='weighted')
recall = recall_score(y_test, y_pred_lasso.round(), average='weighted')

print(f'RMSE: {rmse}')
print(f'Accuracy: {accuracy}')
print(f'F1 Score: {f1score}')
print(f'Precision: {precison}')
print(f'Recall: {recall}')

RMSE: 19.470775538182338
Accuracy: 0.01744846615809122
F1 Score: 0.01167561725125253
Precision: 0.012946651589462009
Recall: 0.01744846615809122


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [51]:
test_pred = lasso_model.predict(test_df)
test_pred = np.clip(test_pred, 0, 100)
print("Min prediction:", test_pred.min())
print("Max prediction:", test_pred.max())
submission_df = pd.DataFrame({
    'id' : test_df_1['id'],
    'target' : test_pred
})
submission_df.to_csv('lasso_submission.csv', index=False)

Min prediction: 0.0
Max prediction: 100.0


Ridge Regression Model

In [49]:
ridge_model = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Ridge(alpha=1.0, random_state=42))
])

ridge_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['album_name_length', 'artist_count', 'album_component_count',
       'key_variety', 'duration_consistency', 'tempo_volatility', 'year',
       'month', 'emotional_charge_mean', 'groove_efficiency_mean',
       'beat_frequency_me...
       'time_signature_mean'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['composition_label_0', 'composition_label_1', 'weekday_of_release',
       'season_of_release', 'lunar_phase', 'creator_collective',
       'composition_label_2'],
      dtype='object'))])),
                ('model', Ridge(random_state=42))])

In [50]:
y_pred_ridge = ridge_model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
accuracy = accuracy_score(y_test, y_pred_ridge.round())
f1score = f1_score(y_test, y_pred_ridge.round(), average='weighted')
precison = precision_score(y_test, y_pred_ridge.round(), average='weighted')
recall = recall_score(y_test, y_pred_ridge.round(), average='weighted')

print(f'RMSE: {rmse}')
print(f'Accuracy: {accuracy}')
print(f'F1 Score: {f1score}')
print(f'Precision: {precison}')
print(f'Recall: {recall}')

RMSE: 10.176678918332481
Accuracy: 0.18487258561921766
F1 Score: 0.18815854155584807
Precision: 0.20647400153035322
Recall: 0.18487258561921766


/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/ravindu_pathirana/Library/Python/3.9/lib/python/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
print("Min prediction:", y_pred.min())
print("Max prediction:", y_pred.max())

Min prediction: 0.0
Max prediction: 100.0


In [39]:
test_pred = ridge_model.predict(test_df)
test_pred = np.clip(test_pred, 0, 100)

In [40]:
print("Min prediction:", test_pred.min())
print("Max prediction:", test_pred.max())

Min prediction: 0.0
Max prediction: 100.0


In [41]:
submission_df = pd.DataFrame({
    'id' : test_df_1['id'],
    'target' : test_pred
})
submission_df.to_csv('ridge_submission.csv', index=False)